In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/models/rsynkr/model-3/pytorch/default/1/model_3_best.pth
/kaggle/input/models/rsynkr/model-2/pytorch/default/1/model_2_best.pth
/kaggle/input/models/rsynkr/model-1/pytorch/default/1/model_1_best.pth
/kaggle/input/datasets/fantineh/next-day-wildfire-spread/next_day_wildfire_spread_train_14.tfrecord
/kaggle/input/datasets/fantineh/next-day-wildfire-spread/next_day_wildfire_spread_train_11.tfrecord
/kaggle/input/datasets/fantineh/next-day-wildfire-spread/next_day_wildfire_spread_eval_01.tfrecord
/kaggle/input/datasets/fantineh/next-day-wildfire-spread/next_day_wildfire_spread_train_08.tfrecord
/kaggle/input/datasets/fantineh/next-day-wildfire-spread/next_day_wildfire_spread_train_13.tfrecord
/kaggle/input/datasets/fantineh/next-day-wildfire-spread/next_day_wildfire_spread_train_01.tfrecord
/kaggle/input/datasets/fantineh/next-day-wildfire-spread/next_day_wildfire_spread_test_00.tfrecord
/kaggle/input/datasets/fantineh/next-day-wildfire-spread/next_day_wildfire_spread_train_1

In [2]:
# This sets up the evaluation environment, dataset paths, exact model paths, and GPU device.

import os
import glob
import numpy as np
import tensorflow as tf
import torch
import torch.nn as nn
from torch.utils.data import IterableDataset, DataLoader
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_DIR = "/kaggle/input/datasets/fantineh/next-day-wildfire-spread"

MODEL_1_PATH = "/kaggle/input/models/rsynkr/model-1/pytorch/default/1/model_1_best.pth"
MODEL_2_PATH = "/kaggle/input/models/rsynkr/model-2/pytorch/default/1/model_2_best.pth"
MODEL_3_PATH = "/kaggle/input/models/rsynkr/model-3/pytorch/default/1/model_3_best.pth"

print("Device:", device)
print("Data directory:", os.path.exists(DATA_DIR))
print("Model 1:", os.path.exists(MODEL_1_PATH))
print("Model 2:", os.path.exists(MODEL_2_PATH))
print("Model 3:", os.path.exists(MODEL_3_PATH))

Device: cuda
Data directory: True
Model 1: True
Model 2: True
Model 3: True


In [3]:
# This recreates the exact U-Net architecture used during training and loads
# the three saved best checkpoints onto the GPU.

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1):
        super().__init__()

        self.enc1 = DoubleConv(in_channels, 32)
        self.enc2 = DoubleConv(32, 64)
        self.enc3 = DoubleConv(64, 128)

        self.pool = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(128, 256)

        self.up3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec3 = DoubleConv(256, 128)

        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec2 = DoubleConv(128, 64)

        self.up1 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec1 = DoubleConv(64, 32)

        self.final = nn.Conv2d(32, out_channels, 1)

    def forward(self, x):
        e1 = self.enc1(x)

        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))

        b = self.bottleneck(self.pool(e3))

        d3 = self.up3(b)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        return self.final(d1)


model_1_eval = UNet(in_channels=1, out_channels=1).to(device)
model_2_eval = UNet(in_channels=8, out_channels=1).to(device)
model_3_eval = UNet(in_channels=13, out_channels=1).to(device)

model_1_eval.load_state_dict(
    torch.load(MODEL_1_PATH, map_location=device)
)

model_2_eval.load_state_dict(
    torch.load(MODEL_2_PATH, map_location=device)
)

model_3_eval.load_state_dict(
    torch.load(MODEL_3_PATH, map_location=device)
)

model_1_eval.eval()
model_2_eval.eval()
model_3_eval.eval()

print("Model 1 loaded successfully")
print("Model 2 loaded successfully")
print("Model 3 loaded successfully")

Model 1 loaded successfully
Model 2 loaded successfully
Model 3 loaded successfully


In [4]:
# This defines the TFRecord files, feature parser, normalization statistics,
# cleaning rules, and preprocessing used by all three models.

TRAIN_FILES = [
    os.path.join(DATA_DIR, f"next_day_wildfire_spread_train_{i:02d}.tfrecord")
    for i in range(15)
]

EVAL_FILES = [
    os.path.join(DATA_DIR, f"next_day_wildfire_spread_eval_{i:02d}.tfrecord")
    for i in range(2)
]

TEST_FILES = [
    os.path.join(DATA_DIR, f"next_day_wildfire_spread_test_{i:02d}.tfrecord")
    for i in range(2)
]

FEATURE_DESCRIPTION = {
    feature: tf.io.FixedLenFeature([4096], tf.float32)
    for feature in [
        "tmmn", "tmmx", "pr", "NDVI", "pdsi", "erc",
        "elevation", "population", "sph", "vs", "th",
        "PrevFireMask", "FireMask"
    ]
}


def parse_example(example):
    return tf.io.parse_single_example(
        example,
        FEATURE_DESCRIPTION
    )


normalization_stats = {
    "tmmn": {"mean": 281.852110, "std": 18.497169},
    "tmmx": {"mean": 297.716643, "std": 19.458073},
    "pr": {"mean": 0.324562, "std": 1.532254},
    "NDVI": {"mean": 5355.635208, "std": 2165.629625},
    "pdsi": {"mean": -0.772871, "std": 2.440719},
    "erc": {"mean": 53.469103, "std": 25.097984},
    "elevation": {"mean": 896.571320, "std": 842.610351},
    "population": {"mean": 30.460330, "std": 214.200051},
    "sph": {"mean": 0.006526, "std": 0.003736},
    "vs": {"mean": 3.627848, "std": 1.309215}
}


def preprocess_features(features):
    features = features.copy()

    features["pr"] = np.maximum(features["pr"], 0.0)
    features["NDVI"] = np.clip(features["NDVI"], -2000.0, 10000.0)
    features["th"] = np.clip(features["th"], -360.0, 360.0)

    for feature, stats in normalization_stats.items():
        features[feature] = (
            features[feature] - stats["mean"]
        ) / stats["std"]

    angle = np.deg2rad(features["th"])

    features["th_sin"] = np.sin(angle)
    features["th_cos"] = np.cos(angle)

    return features


print("Train files:", len(TRAIN_FILES))
print("Validation files:", len(EVAL_FILES))
print("Test files:", len(TEST_FILES))

print("First validation file:")
print(EVAL_FILES[0])

print("\nFirst test file:")
print(TEST_FILES[0])

Train files: 15
Validation files: 2
Test files: 2
First validation file:
/kaggle/input/datasets/fantineh/next-day-wildfire-spread/next_day_wildfire_spread_eval_00.tfrecord

First test file:
/kaggle/input/datasets/fantineh/next-day-wildfire-spread/next_day_wildfire_spread_test_00.tfrecord


In [5]:
# This creates the PyTorch IterableDataset used for validation and test evaluation.
# It matches the input construction used during the original model training.

class WildfireDataset(IterableDataset):

    def __init__(self, files, input_features):
        self.files = files
        self.input_features = input_features

    def __iter__(self):

        for file_path in self.files:

            dataset = tf.data.TFRecordDataset(file_path)

            for raw_example in dataset:

                example = parse_example(raw_example)

                features = {
                    name: example[name].numpy()
                    for name in [
                        "tmmn", "tmmx", "pr", "NDVI", "pdsi",
                        "erc", "elevation", "population",
                        "sph", "vs", "th",
                        "PrevFireMask", "FireMask"
                    ]
                }

                target_raw = features["FireMask"].copy()

                features = preprocess_features(features)

                channels = []

                for feature in self.input_features:

                    if feature == "PrevFireMask":

                        x = features["PrevFireMask"].copy()
                        x[x == -1] = 0
                        channels.append(x)

                    elif feature == "th":

                        channels.append(features["th_sin"])
                        channels.append(features["th_cos"])

                    else:

                        channels.append(features[feature])

                X = np.stack(channels, axis=0)
                X = X.reshape(X.shape[0], 64, 64)

                valid = (target_raw != -1).reshape(64, 64)

                Y = (target_raw == 1).astype(np.float32)
                Y = Y.reshape(64, 64)

                X = torch.tensor(X, dtype=torch.float32)
                Y = torch.tensor(Y[None, :, :], dtype=torch.float32)
                valid = torch.tensor(
                    valid[None, :, :],
                    dtype=torch.bool
                )

                yield X, Y, valid


MODEL_1_FEATURES = [
    "PrevFireMask"
]

MODEL_2_FEATURES = [
    "PrevFireMask",
    "tmmn",
    "tmmx",
    "pr",
    "sph",
    "vs",
    "th"
]

MODEL_3_FEATURES = [
    "PrevFireMask",
    "tmmn",
    "tmmx",
    "pr",
    "sph",
    "vs",
    "th",
    "NDVI",
    "pdsi",
    "erc",
    "elevation",
    "population"
]


val_loader_1 = DataLoader(
    WildfireDataset(EVAL_FILES, MODEL_1_FEATURES),
    batch_size=16,
    num_workers=0
)

val_loader_2 = DataLoader(
    WildfireDataset(EVAL_FILES, MODEL_2_FEATURES),
    batch_size=16,
    num_workers=0
)

val_loader_3 = DataLoader(
    WildfireDataset(EVAL_FILES, MODEL_3_FEATURES),
    batch_size=16,
    num_workers=0
)


test_loader_1 = DataLoader(
    WildfireDataset(TEST_FILES, MODEL_1_FEATURES),
    batch_size=16,
    num_workers=0
)

test_loader_2 = DataLoader(
    WildfireDataset(TEST_FILES, MODEL_2_FEATURES),
    batch_size=16,
    num_workers=0
)

test_loader_3 = DataLoader(
    WildfireDataset(TEST_FILES, MODEL_3_FEATURES),
    batch_size=16,
    num_workers=0
)


print("Evaluation datasets ready.")
print("Model 1 channels:", len(MODEL_1_FEATURES))
print("Model 2 channels:", 8)
print("Model 3 channels:", 13)

Evaluation datasets ready.
Model 1 channels: 1
Model 2 channels: 8
Model 3 channels: 13


In [6]:
# This evaluates a model globally by accumulating TP, FP, FN, and TN
# across the entire dataset while ignoring unknown target pixels.

@torch.no_grad()
def evaluate_global(model, loader, device, threshold=0.5):

    model.eval()

    total_tp = 0
    total_fp = 0
    total_fn = 0
    total_tn = 0

    for X, Y, valid in tqdm(loader, desc="Evaluating"):

        X = X.to(device, non_blocking=True)
        Y = Y.to(device, non_blocking=True)
        valid = valid.to(device, non_blocking=True)

        logits = model(X)
        probs = torch.sigmoid(logits)

        predictions = probs >= threshold

        mask = valid.bool()

        y_true = Y[mask].bool()
        y_pred = predictions[mask]

        total_tp += (y_pred & y_true).sum().item()
        total_fp += (y_pred & ~y_true).sum().item()
        total_fn += (~y_pred & y_true).sum().item()
        total_tn += (~y_pred & ~y_true).sum().item()

    precision = total_tp / max(total_tp + total_fp, 1)
    recall = total_tp / max(total_tp + total_fn, 1)

    dice = (
        2 * total_tp /
        max(2 * total_tp + total_fp + total_fn, 1)
    )

    iou = (
        total_tp /
        max(total_tp + total_fp + total_fn, 1)
    )

    accuracy = (
        (total_tp + total_tn) /
        max(total_tp + total_tn + total_fp + total_fn, 1)
    )

    return {
        "precision": precision,
        "recall": recall,
        "dice": dice,
        "iou": iou,
        "accuracy": accuracy,
        "TP": total_tp,
        "FP": total_fp,
        "FN": total_fn,
        "TN": total_tn
    }

In [7]:
# This performs a quick validation check on Model 1 at threshold 0.5
# to confirm the evaluation pipeline is working correctly.

model_1_val_check = evaluate_global(
    model_1_eval,
    val_loader_1,
    device,
    threshold=0.5
)

print("Model 1 Validation Results")
print("---------------------------")

for key, value in model_1_val_check.items():
    if isinstance(value, float):
        print(f"{key:10s}: {value:.4f}")
    else:
        print(f"{key:10s}: {value}")

Evaluating: 0it [00:00, ?it/s]

I0000 00:00:1791818999.439565      22 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13724 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1791818999.443619      22 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model 1 Validation Results
---------------------------
precision : 0.1633
recall    : 0.6202
dice      : 0.2585
iou       : 0.1484
accuracy  : 0.9513
TP        : 63633
FP        : 326081
FN        : 38961
TN        : 7071710


In [8]:
# This searches for the Model 1 threshold that maximizes global validation Dice.

thresholds_1 = np.arange(0.10, 1.00, 0.02)

results_1 = []

for threshold in tqdm(thresholds_1, desc="Model 1 threshold search"):

    metrics = evaluate_global(
        model_1_eval,
        val_loader_1,
        device,
        threshold=float(threshold)
    )

    results_1.append({
        "threshold": float(threshold),
        "dice": metrics["dice"],
        "iou": metrics["iou"],
        "precision": metrics["precision"],
        "recall": metrics["recall"]
    })

best_1 = max(results_1, key=lambda x: x["dice"])

print("\nModel 1 Best Validation Threshold")
print("----------------------------------")
print(f"Threshold : {best_1['threshold']:.2f}")
print(f"Dice      : {best_1['dice']:.4f}")
print(f"IoU       : {best_1['iou']:.4f}")
print(f"Precision : {best_1['precision']:.4f}")
print(f"Recall    : {best_1['recall']:.4f}")

Model 1 threshold search:   0%|          | 0/45 [00:00<?, ?it/s]

Evaluating: 0it [00:00, ?it/s]

Evaluating: 0it [00:00, ?it/s]

Evaluating: 0it [00:00, ?it/s]

Evaluating: 0it [00:00, ?it/s]

Evaluating: 0it [00:00, ?it/s]

... (37 outputs left out)

Evaluating: 0it [00:00, ?it/s]

Evaluating: 0it [00:00, ?it/s]

Evaluating: 0it [00:00, ?it/s]


Model 1 Best Validation Threshold
----------------------------------
Threshold : 0.94
Dice      : 0.3087
IoU       : 0.1825
Precision : 0.2354
Recall    : 0.4481


In [9]:
# This searches for the Model 2 threshold that maximizes global validation Dice.
# The test set remains completely untouched.

thresholds_2 = np.arange(0.10, 1.00, 0.02)

results_2 = []

for threshold in tqdm(thresholds_2, desc="Model 2 threshold search"):

    metrics = evaluate_global(
        model_2_eval,
        val_loader_2,
        device,
        threshold=float(threshold)
    )

    results_2.append({
        "threshold": float(threshold),
        "dice": metrics["dice"],
        "iou": metrics["iou"],
        "precision": metrics["precision"],
        "recall": metrics["recall"]
    })

best_2 = max(results_2, key=lambda x: x["dice"])

print("\nModel 2 Best Validation Threshold")
print("----------------------------------")
print(f"Threshold : {best_2['threshold']:.2f}")
print(f"Dice      : {best_2['dice']:.4f}")
print(f"IoU       : {best_2['iou']:.4f}")
print(f"Precision : {best_2['precision']:.4f}")
print(f"Recall    : {best_2['recall']:.4f}")

Model 2 threshold search:   0%|          | 0/45 [00:00<?, ?it/s]

Evaluating: 0it [00:00, ?it/s]

Evaluating: 0it [00:00, ?it/s]

Evaluating: 0it [00:00, ?it/s]

Evaluating: 0it [00:00, ?it/s]

Evaluating: 0it [00:00, ?it/s]

... (37 outputs left out)

Evaluating: 0it [00:00, ?it/s]

Evaluating: 0it [00:00, ?it/s]

Evaluating: 0it [00:00, ?it/s]


Model 2 Best Validation Threshold
----------------------------------
Threshold : 0.94
Dice      : 0.3250
IoU       : 0.1940
Precision : 0.2692
Recall    : 0.4098


In [10]:
# This performs the final evaluation on the untouched test set using the
# thresholds selected exclusively from the validation set.

test_results_1 = evaluate_global(
    model_1_eval,
    test_loader_1,
    device,
    threshold=0.94
)

test_results_2 = evaluate_global(
    model_2_eval,
    test_loader_2,
    device,
    threshold=0.94
)

test_results_3 = evaluate_global(
    model_3_eval,
    test_loader_3,
    device,
    threshold=0.93
)

print("\nMODEL 1 — TEST RESULTS")
print("----------------------")
for key, value in test_results_1.items():
    print(f"{key:10s}: {value:.4f}" if isinstance(value, float) else f"{key:10s}: {value}")

print("\nMODEL 2 — TEST RESULTS")
print("----------------------")
for key, value in test_results_2.items():
    print(f"{key:10s}: {value:.4f}" if isinstance(value, float) else f"{key:10s}: {value}")

print("\nMODEL 3 — TEST RESULTS")
print("----------------------")
for key, value in test_results_3.items():
    print(f"{key:10s}: {value:.4f}" if isinstance(value, float) else f"{key:10s}: {value}")

Evaluating: 0it [00:00, ?it/s]

Evaluating: 0it [00:00, ?it/s]

Evaluating: 0it [00:00, ?it/s]


MODEL 1 — TEST RESULTS
----------------------
precision : 0.2640
recall    : 0.5503
dice      : 0.3569
iou       : 0.2172
accuracy  : 0.9751
TP        : 46406
FP        : 129347
FN        : 37925
TN        : 6514021

MODEL 2 — TEST RESULTS
----------------------
precision : 0.3200
recall    : 0.4618
dice      : 0.3780
iou       : 0.2331
accuracy  : 0.9810
TP        : 38948
FP        : 82778
FN        : 45383
TN        : 6560590

MODEL 3 — TEST RESULTS
----------------------
precision : 0.2830
recall    : 0.5304
dice      : 0.3691
iou       : 0.2263
accuracy  : 0.9773
TP        : 44733
FP        : 113326
FN        : 39598
TN        : 6530042
